# 2 · Traversal — filters, hop counts, direction, `OPTIONAL`

One `Start`, then one `Hop` per step. This notebook is the whole read API, and it
ends with the four invariants the engine is built around — each one written as a
runnable assertion, because each one is a bug this library actually had.

Uses the same seven-node demo graph as notebook 01 (see
[`demo_graph.py`](demo_graph.py)).

In [1]:
from demo_graph import arrows, connect, names, seed
from hopai import AND, BETWEEN, GT, NOT, OR, Hop, Start

graph = connect("nb_02_traversal")
seed(graph)
print(names(graph.traverse(Start())))

['Acme', 'Alice', 'Bob', 'Carol', 'Dave', 'Erin', 'Globex']


## `where` and `via`

Two filters per hop, and they filter different things:

- **`where`** — the node this hop *lands on*.
- **`via`** — every edge this hop *walks over*.

Both take the same filter language, which is the rest of this section.

### A dict is AND-of-its-keys; a list value is OR-of-values

In [2]:
print(names(graph.traverse(Start(where={"type": "person"}))))
print(names(graph.traverse(Start(where={"type": "person", "active": True}))))
print(names(graph.traverse(Start(where={"type": ["person", "company"]}))))   # IN-like

['Alice', 'Bob', 'Carol', 'Dave', 'Erin']
['Alice', 'Bob', 'Carol', 'Erin']
['Acme', 'Alice', 'Bob', 'Carol', 'Dave', 'Erin', 'Globex']


Equality is JSONB **containment** (`@>`), not a `->>` string comparison — which is
why it uses the GIN index, and why `{"active": True}` matches the JSON boolean
rather than the string `"true"`.

### `OR`, `AND`, `NOT` are explicit classes

A plain dict already means AND. Anything else has to say so, so that what a filter
means is visible from what you wrote rather than inferred from which Python type
you reached for.

In [3]:
print(names(graph.traverse(Start(where=OR({"type": "company"}, {"name": "Alice"})))))
print(names(graph.traverse(Start(where=AND(OR({"city": "Berlin"}, {"city": "Lisbon"}),
                                           {"active": True})))))

['Acme', 'Alice', 'Globex']
['Alice', 'Bob', 'Carol']


### `NOT` keeps the rows that are missing the key

This is the one filter whose semantics you have to know, and it is a deliberate
choice rather than an accident. `NOT` negates a *containment test*, so a node with
no `city` key at all fails the positive test and therefore **passes** the negation.

Erin has no `city`. She is in the answer:

In [4]:
not_berlin = graph.traverse(Start(where=AND({"type": "person"}, NOT({"city": "Berlin"}))))
print(names(not_berlin))
assert "Erin" in names(not_berlin)

['Carol', 'Dave', 'Erin']


Naive negation — `properties->>'city' <> 'Berlin'` — evaluates to `NULL` for Erin,
and SQL drops the row. Same English sentence, quietly different answer. Cypher's
own `NOT x.city = 'Berlin'` has the same hole, which is why hopai's Cypher front
end *refuses* `<>` rather than translating it (notebook 04).

Here is that difference, run side by side through the escape hatch:

In [5]:
naive = graph.traverse(Start(where=AND(
    {"type": "person"},
    lambda col: col.op("->>")("city") != "Berlin",     # raw SQL semantics
)))
print("containment NOT:", names(not_berlin))
print("naive  <>     :", names(naive), "<- Erin silently gone")

containment NOT: ['Carol', 'Dave', 'Erin']
naive  <>     : ['Carol', 'Dave'] <- Erin silently gone


That `lambda` is the documented escape hatch: any callable receiving the real
`properties` column and returning a real SQLAlchemy boolean. Use it for the things
the DSL does not cover — a regex, say — not to route around the DSL.

In [6]:
starts_with_a_or_b = graph.traverse(Start(where=lambda col: col.op("->>")("name").op("~")("^[AB]")))
print(names(starts_with_a_or_b))

['Acme', 'Alice', 'Bob']


### A bare list raises

`[{"a": 1}, {"b": 2}]` reads as "both of these" to a human and would have meant OR.
It is refused, and the error names the fix — a pattern worth noticing, because
every refusal in this library is written that way.

In [7]:
try:
    graph.traverse(Start(where=[{"type": "person"}, {"type": "company"}]))
except TypeError as exc:
    print(f"TypeError: {exc}")

TypeError: a bare list is ambiguous -- use OR(...) to mean 'any of these filters', e.g. OR({'type': 'person'}, {'type': 'company'}) instead of [{'type': 'person'}, {'type': 'company'}]


### Comparisons

In [8]:
print(names(graph.traverse(Start(where=GT("age", 40)))))
print(names(graph.traverse(Start(where=BETWEEN("age", 25, 45)))))

['Bob', 'Dave']
['Alice', 'Bob', 'Carol']


## Hop counts

`hops=N` is exactly N edges; `hops=(min, max)` is a range. The default is 1.

In [9]:
for hops in (1, 2, (1, 2), 3):
    result = graph.traverse(Start(where={"name": "Alice"}), Hop(via={"kind": "friend"}, hops=hops))
    print(f"hops={str(hops):8} {names(result)}")

hops=1        ['Alice', 'Bob', 'Carol']
hops=2        ['Alice', 'Bob', 'Carol', 'Dave']
hops=(1, 2)   ['Alice', 'Bob', 'Carol', 'Dave']
hops=3        ['Alice', 'Bob', 'Carol', 'Dave', 'Erin']


Read the `hops=2` line carefully: it reports Alice, Bob, Carol *and* Dave — four
nodes for a hop that lands only on Dave. A hop spanning several edges reports
**every real edge it walked**, not one fabricated edge between the endpoints, and
the intermediate nodes come back with them.

In [10]:
two = graph.traverse(Start(where={"name": "Alice"}), Hop(via={"kind": "friend"}, hops=2))
for arrow in arrows(two):
    print(arrow)

Alice -friend-> Bob
Alice -friend-> Carol
Bob -friend-> Dave
Carol -friend-> Dave


### Cycles terminate

`Erin -friend-> Alice` closes a loop. The recursive walk carries a path array and
refuses to revisit a node within the same hop, so a wide range is safe:

In [11]:
for start in ("Alice", "Bob"):
    wide = graph.traverse(Start(where={"name": start}), Hop(via={"kind": "friend"}, hops=(1, 20)))
    print(f"from {start}:")
    for arrow in arrows(wide):
        print("   ", arrow)

from Alice:
    Alice -friend-> Bob
    Alice -friend-> Carol
    Bob -friend-> Dave
    Carol -friend-> Dave
    Dave -friend-> Erin
from Bob:
    Alice -friend-> Carol
    Bob -friend-> Dave
    Dave -friend-> Erin
    Erin -friend-> Alice


A range of 20 over a graph with a loop in it finishes, and finishes with each edge
reported once. Look at what the path array actually does, though — the two walks
above are not the same set of edges:

- From **Alice**, `Erin -friend-> Alice` is missing. Closing the loop would revisit
  the node this walk started from, and a walk never revisits a node it has already
  been to.
- From **Bob**, that same edge *is* there — Alice is new to this walk — and the walk
  carries on past her to Carol before stopping at Bob.

So "cycle protection" here means *no node twice within one hop's walk*, not "cycles
are pruned out of the data". The array that enforces it is carried on every
recursive row: cheap at moderate depth, measurably not cheap past roughly ten hops
on a single-segment traversal (`benchmarks/` has the numbers rather than a guess).

## Direction

Forward follows `start_id → end_id`. Backward follows `end_id → start_id`, which is
how you ask *"what points at this?"*

In [12]:
# "Who works at Acme?" -- start at the company, walk works_at backwards.
at_acme = graph.traverse(
    Start(where={"name": "Acme"}),
    Hop(via={"kind": "works_at"}, direction="backward", where={"type": "person"}),
)
print(names(at_acme))

['Acme', 'Bob', 'Carol']


Direction is **per hop**, so one chain can mix both. "Who are Bob's colleagues?" is
out to his employer and back down to everyone else there:

In [13]:
colleagues = graph.traverse(
    Start(where={"name": "Bob"}),
    Hop(via={"kind": "works_at"}),                          # up to the company
    Hop(via={"kind": "works_at"}, direction="backward"),    # back down to its people
)
print(names(colleagues))     # Bob is his own colleague here -- filter him out yourself

['Acme', 'Bob', 'Carol']


## `OPTIONAL`

Cypher's `OPTIONAL MATCH`: keep the nodes that reached this point in the chain even
when this hop finds nothing for them.

Without it, "every active person and the company they work for" quietly becomes
"every active person **who works somewhere**":

In [14]:
required = graph.traverse(
    Start(where={"type": "person", "active": True}),
    Hop(via={"kind": "works_at"}, where={"type": "company"}),
)
optional = graph.traverse(
    Start(where={"type": "person", "active": True}),
    Hop(via={"kind": "works_at"}, where={"type": "company"}, optional=True),
)
print("required:", names(required))
print("optional:", names(optional), "<- Alice and Erin are back, with no company attached")

required: ['Acme', 'Bob', 'Carol']
optional: ['Acme', 'Alice', 'Bob', 'Carol', 'Erin'] <- Alice and Erin are back, with no company attached


`optional=True` is valid **only on the last hop**. Mid-chain it would mean every
downstream hop tolerating a missing anchor — a materially larger feature than a
flag, and one this library has not built. So it raises instead of half-working:

In [15]:
try:
    graph.traverse(Start(), Hop(optional=True), Hop())
except ValueError as exc:
    print(f"ValueError: {exc}")

ValueError: hop 0 (unlabeled): optional=True is only supported on the LAST hop in a chain. If you need multiple optional extensions, run separate queries.


## The four invariants

Everything above rests on these. Each was a real bug; each has a test in
`tests/test_hopai.py` named after it. They are worth running here because they are
the behaviours most likely to be broken by a "harmless" optimization.

**1 · Fan-in is preserved.** Two parents feeding one intermediate node must both be
reported. An earlier engine kept one global path per destination node and silently
dropped the second.

In [16]:
fan_in = graph.traverse(Start(where={"name": "Alice"}), Hop(via={"kind": "friend"}, hops=(1, 2)))
walked = arrows(fan_in)
assert "Bob -friend-> Dave" in walked and "Carol -friend-> Dave" in walked, walked
print("\n".join(walked))

Alice -friend-> Bob
Alice -friend-> Carol
Bob -friend-> Dave
Carol -friend-> Dave


**2 · A multi-edge hop reports every edge it walked** — no fabricated shortcut
between the endpoints.

In [17]:
three = graph.traverse(Start(where={"name": "Alice"}), Hop(via={"kind": "friend"}, hops=3))
assert "Dave -friend-> Erin" in arrows(three)          # the third edge, really walked
assert "Alice -friend-> Erin" not in arrows(three)     # not a shortcut
print(arrows(three))

['Alice -friend-> Bob', 'Alice -friend-> Carol', 'Bob -friend-> Dave', 'Carol -friend-> Dave', 'Dave -friend-> Erin']


**3 · Reported nodes derive from the edges found, never from the seed set** — which
is what prunes dead ends, with no separate pruning pass. Alice seeds the query
below and does not survive it, because she has no `works_at` edge.

In [18]:
employed = graph.traverse(Start(where={"type": "person"}), Hop(via={"kind": "works_at"}))
assert "Alice" not in names(employed)
print(names(employed))

['Acme', 'Bob', 'Carol', 'Dave', 'Globex']


**4 · Each hop reports what *it* matched.** The corollary of invariant 3, and the
one that surprises people: a branch that satisfies hop 1 contributes its hop-1
edges even when hop 2 finds nothing to continue with. Erin matches hop 1 below and
works nowhere, and `Dave -friend-> Erin` is in the result anyway.

If you need "only the chains that made it all the way", filter the result by what
you actually asked for — the last hop's `where` — rather than assuming the subgraph
is the set of complete chains.

In [19]:
chain = graph.traverse(
    Start(where={"name": "Alice"}),
    Hop(via={"kind": "friend"}, hops=(1, 4), where={"active": True}),
    Hop(via={"kind": "works_at"}, where={"type": "company"}),
)
print(arrows(chain))
assert "Dave -friend-> Erin" in arrows(chain)

companies = [n["properties"]["name"] for n in chain.nodes if n["properties"]["type"] == "company"]
print("the answer asked for:", companies)

['Alice -friend-> Bob', 'Alice -friend-> Carol', 'Bob -friend-> Dave', 'Bob -works_at-> Acme', 'Carol -friend-> Dave', 'Carol -works_at-> Acme', 'Dave -friend-> Erin']
the answer asked for: ['Acme']


---

Next: [03 · Aggregation](03_aggregation.ipynb) — a number instead of a subgraph,
computed in the database.